## Notebook 07 — Baseline LightGBM Model

---

This notebook:
1. Loads + merges data, drops >80% missing columns, removes `TransactionID`
2. Performs stratified 80/20 train-validation split
3. Converts object columns to 'category' dtype (for LightGBM compatibility)
4. Trains baseline LightGBM classifier with early stopping
5. Evaluates on validation set (ROC-AUC, PR-AUC)
6. Displays best iteration and top 20 feature importances

> **No imputation or encoding is performed. Only dtype conversion for categorical features.**

---
## 📦 Step 0 — Import Libraries, Load & Split Data

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../Datasets/'
TARGET    = 'isFraud'

# Load & merge
print('Loading and merging...')
train_transaction = pd.read_csv(DATA_PATH + 'train_transaction.csv')
train_identity    = pd.read_csv(DATA_PATH + 'train_identity.csv')
train_raw = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
del train_transaction, train_identity

# Drop >80% missing columns
missing_pct  = train_raw.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
train = train_raw.drop(columns=cols_to_drop)
del train_raw

# Drop identifier column
train = train.drop(columns=['TransactionID'], errors='ignore')

# Stratified 80/20 split
X = train.drop(columns=[TARGET])
y = train[TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
del train, X, y

print(f'✅ X_train shape : {X_train.shape}')
print(f'✅ X_val shape   : {X_val.shape}')
print(f'✅ Train fraud % : {y_train.mean()*100:.2f}%')
print(f'✅ Val fraud %   : {y_val.mean()*100:.2f}%')

Loading and merging...
✅ X_train shape : (472432, 358)
✅ X_val shape   : (118108, 358)
✅ Train fraud % : 3.50%
✅ Val fraud %   : 3.50%


---
## 🔄 Step 1 — Convert Object Columns to Category Dtype

In [3]:
# Identify object columns
object_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f'Found {len(object_cols)} object columns')
print(f'Converting to category dtype...')
print()

# Convert to category dtype in both train and validation sets
for col in object_cols:
    X_train[col] = X_train[col].astype('category')
    X_val[col] = X_val[col].astype('category')

print(f'✅ Converted {len(object_cols)} categorical columns in X_train and X_val')
print()
print('Categorical columns:')
for i, col in enumerate(object_cols, 1):
    print(f'  {i:2d}. {col}')

Found 26 object columns
Converting to category dtype...

✅ Converted 26 categorical columns in X_train and X_val

Categorical columns:
   1. ProductCD
   2. card4
   3. card6
   4. P_emaildomain
   5. R_emaildomain
   6. M1
   7. M2
   8. M3
   9. M4
  10. M5
  11. M6
  12. M7
  13. M8
  14. M9
  15. id_12
  16. id_15
  17. id_16
  18. id_28
  19. id_29
  20. id_31
  21. id_35
  22. id_36
  23. id_37
  24. id_38
  25. DeviceType
  26. DeviceInfo


---
## 🚀 Step 2 — Initialize LightGBM Model

In [4]:
# Initialize LightGBM classifier with specified parameters
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    scale_pos_weight=27.6,
    random_state=42,
    verbose=-1
)

print('✅ LightGBM model initialized')
print(f'   Objective: binary')
print(f'   Metric: auc')
print(f'   Max estimators: 1000')
print(f'   Learning rate: 0.05')
print(f'   Num leaves: 64')
print(f'   Scale pos weight: 27.6')
print(f'   Early stopping rounds: 100')

✅ LightGBM model initialized
   Objective: binary
   Metric: auc
   Max estimators: 1000
   Learning rate: 0.05
   Num leaves: 64
   Scale pos weight: 27.6
   Early stopping rounds: 100


---
## 🎯 Step 3 — Train Model with Early Stopping

In [5]:
print('Training LightGBM model...')
print()

# Train with early stopping
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(period=50)
    ]
)

print()
print('✅ Training complete')

Training LightGBM model...

Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.92496
[100]	valid_0's auc: 0.938345
[150]	valid_0's auc: 0.945205
[200]	valid_0's auc: 0.949704
[250]	valid_0's auc: 0.952877
[300]	valid_0's auc: 0.955505
[350]	valid_0's auc: 0.957204
[400]	valid_0's auc: 0.959421
[450]	valid_0's auc: 0.960672
[500]	valid_0's auc: 0.962008
[550]	valid_0's auc: 0.963193
[600]	valid_0's auc: 0.96417
[650]	valid_0's auc: 0.964984
[700]	valid_0's auc: 0.965722
[750]	valid_0's auc: 0.966259
[800]	valid_0's auc: 0.966865
[850]	valid_0's auc: 0.96736
[900]	valid_0's auc: 0.967859
[950]	valid_0's auc: 0.968224
[1000]	valid_0's auc: 0.968701
Did not meet early stopping. Best iteration is:
[997]	valid_0's auc: 0.968709

✅ Training complete


---
## 📊 Step 4 — Evaluate on Validation Set

In [6]:
# Get predictions on validation set
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# Compute metrics
roc_auc = roc_auc_score(y_val, y_val_pred_proba)
pr_auc = average_precision_score(y_val, y_val_pred_proba)

# Get best iteration
best_iter = model.best_iteration_

# Print results
print('=' * 60)
print('   VALIDATION METRICS')
print('=' * 60)
print(f'Validation ROC-AUC:        {roc_auc:.6f}')
print(f'Validation PR-AUC:         {pr_auc:.6f}')
print(f'Best Iteration:            {best_iter}')
print('=' * 60)

   VALIDATION METRICS
Validation ROC-AUC:        0.968709
Validation PR-AUC:         0.822145
Best Iteration:            997


---
## 🏆 Step 5 — Top 20 Feature Importances

In [7]:
# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

feature_importance.index += 1
top20 = feature_importance.head(20)

# Print top 20 features
W = 70
sep = '=' * W
dash = '-' * W
hdr = f'{"Rank":<6} {"Feature":<30} {"Importance":>20}'

print(sep)
print('   TOP 20 FEATURES — By Importance (Gain)')
print(sep)
print(hdr)
print(dash)
for rank, row in top20.iterrows():
    print(
        f'{rank:<6} {row["Feature"]:<30} '
        f'{row["Importance"]:>20,.2f}'
    )
print(sep)

   TOP 20 FEATURES — By Importance (Gain)
Rank   Feature                                  Importance
----------------------------------------------------------------------
1      card1                                      3,708.00
2      TransactionDT                              3,701.00
3      card2                                      3,462.00
4      addr1                                      3,457.00
5      TransactionAmt                             3,087.00
6      P_emaildomain                              2,499.00
7      id_31                                      2,324.00
8      D15                                        1,577.00
9      dist1                                      1,499.00
10     DeviceInfo                                 1,487.00
11     C13                                        1,385.00
12     D4                                         1,320.00
13     card5                                      1,244.00
14     D10                                        1,207.00
15